In [1]:
import re, json
from pathlib import Path
from langchain_core.documents import Document

CLEAN = Path("../data/clean")
LAWS = {   # key -> (full title for citations, short name for chunk headers)
    "labour_law":                ("UAE Labour Law (Federal Decree-Law No. 33 of 2021)",        "Labour Law"),
    "exec_regs":                 ("Executive Regulation (Cabinet Resolution No. 1 of 2022)",   "Executive Regulation"),
    "labour_law_amendment_2022": ("Labour Law Amendment (Federal Decree-Law No. 14 of 2022)",  "Labour Law Amendment 2022"),
    "pdpl":                      ("UAE PDPL (Federal Decree-Law No. 45 of 2021)",              "PDPL"),
}
texts = {k: (CLEAN / f"{k}.txt").read_text(encoding="utf-8") for k in LAWS}
for k, t in texts.items():
    print(f"{k:28} {len(t):>7,} chars")

labour_law                    82,351 chars
exec_regs                     53,018 chars
labour_law_amendment_2022      1,605 chars
pdpl                          41,127 chars


In [9]:
# Heading = "Article (n)" or "Article )n (" ALONE on its line ($ = end of line).
# A mention like "Article (5) of this Decree-Law..." continues on the same line, so it won't match.
HEADING = re.compile(r"(?m)^Article\s*[()]?\s*(\d+)\s*[()]?\s*$")

def find_article_starts(text, max_gap=3):
    """Accept numbers that go UP (allowing small gaps), so one odd heading can't break the rest."""
    starts, rejected, last = [], [], 0
    for m in HEADING.finditer(text):
        n = int(m.group(1))
        if last < n <= last + max_gap:
            starts.append((n, m.start()))
            last = n
        else:
            rejected.append((n, text[m.start():m.start() + 60].replace("\n", " ")))
    return starts, rejected

for k, t in texts.items():
    starts, rejected = find_article_starts(t)
    nums = [n for n, _ in starts]
    missing = sorted(set(range(1, nums[-1] + 1)) - set(nums))
    print(f"{k:28} articles: {len(starts):3} (1..{nums[-1]})  missing: {missing}  rejected: {len(rejected)}")

# Diagnostic: lines starting with "Article" that are NOT accepted headings (mentions or odd formats)
print()
for k, t in texts.items():
    odd = [l for l in t.splitlines() if l.startswith("Article") and not HEADING.match(l)]
    print(f"{k:28} other 'Article' lines: {len(odd)}  {odd[:4]}")

labour_law                   articles:  74 (1..74)  missing: []  rejected: 0
exec_regs                    articles:  39 (1..39)  missing: []  rejected: 0
labour_law_amendment_2022    articles:   3 (1..3)  missing: []  rejected: 0
pdpl                         articles:  31 (1..31)  missing: []  rejected: 0

labour_law                   other 'Article' lines: 2  ['Article except at the request of the Minister or his delegate.', 'Article based on the Employer request before a ruling is issued in it, in return for paying an']
exec_regs                    other 'Article' lines: 2  ['Article (5) of the Decree-Law and the provisions of this article in accordance with the', 'Articles (42) and (45) of the Federal Decree-Law.']
labour_law_amendment_2022    other 'Article' lines: 0  []
pdpl                         other 'Article' lines: 0  []


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

MAX_CHARS = 1500    # ~350 tokens; our embedding model (bge-small) reads max 512 tokens
sub_splitter = RecursiveCharacterTextSplitter(chunk_size=MAX_CHARS, chunk_overlap=150)

def article_chunks(key, text):
    title, short = LAWS[key]
    starts, _ = find_article_starts(text)
    chunks = []

    def add(n, body, label):
        parts = sub_splitter.split_text(body) if len(body) > MAX_CHARS else [body]   # split only LONG articles
        for j, part in enumerate(parts, 1):
            tag = f"[{short} — {label}" + (f", part {j}/{len(parts)}" if len(parts) > 1 else "") + "]"
            chunks.append(Document(
                page_content=f"{tag}\n{part}",          # header tells the model WHICH law + article
                metadata={"source": key, "title": title, "article": n, "part": j,
                          "chunk_id": f"{key}-a{n}-p{j}"}))

    if starts and starts[0][1] > 0:                     # text before Article 1 (preamble)
        add(0, text[:starts[0][1]].strip(), "Preamble")
    for i, (n, pos) in enumerate(starts):
        end = starts[i + 1][1] if i + 1 < len(starts) else len(text)
        body = HEADING.sub(f"Article ({n})", text[pos:end].strip(), count=1)   # fix reversed brackets
        add(n, body, f"Article {n}")
    return chunks

art_chunks = [c for k, t in texts.items() for c in article_chunks(k, t)]
print("Total article chunks:", len(art_chunks))
for k in LAWS:
    print(f"  {k:28} {sum(c.metadata['source'] == k for c in art_chunks)}")

print("\n--- Example: Labour Law, Article 29 ---")
for c in art_chunks:
    if c.metadata["source"] == "labour_law" and c.metadata["article"] == 29:
        print(c.metadata, "\n", c.page_content[:700], "\n")

Total article chunks: 204
  labour_law                   97
  exec_regs                    56
  labour_law_amendment_2022    4
  pdpl                         47

--- Example: Labour Law, Article 29 ---
{'source': 'labour_law', 'title': 'UAE Labour Law (Federal Decree-Law No. 33 of 2021)', 'article': 29, 'part': 1, 'chunk_id': 'labour_law-a29-p1'} 
 [Labour Law — Article 29, part 1/2]
Article (29)
Annual leave
1. Without prejudice to the Worker's rights acquired for the period preceding the date of
enforcement of the provisions of this Decree by law, the Worker shall be entitled to an
annual leave with full pay of not less than:
a. Thirty days for each year of his extended service.
b. Two days for each month if his service period is more than six months and less than
one year.
c. A leave for parts of the last year he spent at work, in the event that his service ends
before using his annual leave balance.
2. The part-time Worker is entitled to an annual leave according to the actual work

Step 3, Part B: fixed-size chunks and the comparison

Why build a second version: to show with numbers that article chunks are better, instead of just claiming it. Fixed-size is the common default: cut every N characters, ignoring structure. In Step 8 we’ll run the same test questions on both versions.

In [10]:
fixed_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100, add_start_index=True)

def fixed_chunks(key, text):
    title, _ = LAWS[key]
    starts, _ = find_article_starts(text)
    docs = fixed_splitter.create_documents([text], metadatas=[{"source": key, "title": title}])
    for i, d in enumerate(docs):
        pos = d.metadata["start_index"]                       # where this chunk starts in the text
        d.metadata["article"] = max([n for n, s in starts if s <= pos], default=0)  # article it STARTS in
        d.metadata["chunk_id"] = f"{key}-f{i}"
    return docs

fix_chunks = [c for k, t in texts.items() for c in fixed_chunks(k, t)]
print("Total fixed chunks:", len(fix_chunks))

Total fixed chunks: 262


In [11]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")   # same model we use in Step 4

def stats(name, chunks):
    toks = [len(tok.encode(c.page_content)) for c in chunks]
    print(f"{name:8} chunks: {len(chunks):4} | tokens avg {sum(toks)/len(toks):4.0f}  "
          f"min {min(toks):3}  max {max(toks):4} | over 512 (cut off!): {sum(t > 512 for t in toks)}")

stats("article", art_chunks)
stats("fixed",   fix_chunks)

/Users/abdul/Projects/uae-labour-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


article  chunks:  204 | tokens avg  187  min  28  max  345 | over 512 (cut off!): 0
fixed    chunks:  262 | tokens avg  149  min  49  max  182 | over 512 (cut off!): 0


In [12]:
print("=== ARTICLE chunk ===")
a = next(c for c in art_chunks if c.metadata["source"] == "labour_law" and c.metadata["article"] == 29)
print(a.page_content[:500])

print("\n=== FIXED chunks touching 'annual leave' ===")
for c in [c for c in fix_chunks if c.metadata["source"] == "labour_law" and "annual leave" in c.page_content.lower()][:2]:
    print("---", c.metadata["chunk_id"], "| starts in article", c.metadata["article"])
    print(c.page_content[:500])

=== ARTICLE chunk ===
[Labour Law — Article 29, part 1/2]
Article (29)
Annual leave
1. Without prejudice to the Worker's rights acquired for the period preceding the date of
enforcement of the provisions of this Decree by law, the Worker shall be entitled to an
annual leave with full pay of not less than:
a. Thirty days for each year of his extended service.
b. Two days for each month if his service period is more than six months and less than
one year.
c. A leave for parts of the last year he spent at work, in the e

=== FIXED chunks touching 'annual leave' ===
--- labour_law-f50 | starts in article 27
them.
Article (28)
Official Holidays and Requiring Worker to Work Therein
1. The Worker is entitled to an official leave with full pay on the official holidays to be
determined by a Cabinet resolution.
2. If circumstances of work necessitate that the Worker works in any of the official holidays,
the Employer shall compensate him with another rest day for each day in which he works
durin

In [13]:
OUT = Path("../data/chunks"); OUT.mkdir(exist_ok=True)

def save(chunks, path):
    with open(path, "w", encoding="utf-8") as f:
        for c in chunks:
            f.write(json.dumps({"text": c.page_content, "metadata": c.metadata}, ensure_ascii=False) + "\n")
    print(f"saved {len(chunks)} -> {path}")

save(art_chunks, OUT / "article_chunks.jsonl")
save(fix_chunks, OUT / "fixed_chunks.jsonl")

saved 204 -> ../data/chunks/article_chunks.jsonl
saved 262 -> ../data/chunks/fixed_chunks.jsonl
